In [ ]:

# ============================================
# 3-Phase Induction Motor Fault Analysis
# Environment Verification
# ============================================

import sys
import os

print("Python version:", sys.version)

# Check GPU
!nvidia-smi

# Install initial libraries
!pip install -q numpy pandas scipy matplotlib seaborn scikit-learn xgboost torch tqdm openpyxl

print("\nEnvironment setup completed!")


Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Fri Nov 13 13:34:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |     

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    x = torch.randn(1000, 1000).cuda()
    y = torch.randn(1000, 1000).cuda()

    z = torch.matmul(x, y)

    print("GPU computation successful!")
    print("Output device:", z.device)


PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU computation successful!
Output device: cuda:0


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = "/content/drive/MyDrive/Motor_Fault_Analysis"

RAW_DIR = os.path.join(PROJECT_DIR, "datasets", "raw")
PROCESSED_DIR = os.path.join(PROJECT_DIR, "datasets", "processed")
METADATA_DIR = os.path.join(PROJECT_DIR, "datasets", "metadata")

for folder in [RAW_DIR, PROCESSED_DIR, METADATA_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Project directories created successfully!")
print(PROJECT_DIR)


Mounted at /content/drive
Project directories created successfully!
/content/drive/MyDrive/Motor_Fault_Analysis


In [ ]:

import requests
import pandas as pd

ARTICLE_ID = 27216219

api_url = f"https://api.figshare.com/v2/articles/{ARTICLE_ID}"

response = requests.get(api_url, timeout=30)
response.raise_for_status()

dataset_info = response.json()

print("Dataset:", dataset_info["title"])
print("DOI:", dataset_info["doi"])
print("License:", dataset_info.get("license", {}).get("name"))

files = []

for file in dataset_info["files"]:
    files.append({
        "filename": file["name"],
        "size_MB": round(file["size"] / (1024 * 1024), 2),
        "download_url": file["download_url"]
    })

file_df = pd.DataFrame(files)

display(file_df)

print("Total files:", len(file_df))
print("Total size (MB):", file_df["size_MB"].sum())


Dataset: MOTOR FAULT DETECTION DATA
DOI: 10.6084/m9.figshare.27216219.v1
License: CC BY 4.0


,filename,size_MB,download_url
0,FILE 1.mat,68.66,https://ndownloader.figshare.com/files/53993651
1,FILE 2.mat,68.66,https://ndownloader.figshare.com/files/53993645
2,FILE 3.mat,68.66,https://ndownloader.figshare.com/files/53993648
3,FILE 4.mat,68.66,https://ndownloader.figshare.com/files/53993657
4,FILE 5.mat,68.66,https://ndownloader.figshare.com/files/53993666
5,FILE 6.mat,68.66,https://ndownloader.figshare.com/files/53993642
6,FILE 7.mat,68.66,https://ndownloader.figshare.com/files/53993663
7,FILE 8.mat,68.66,https://ndownloader.figshare.com/files/53993669
8,FILE 9.mat,68.66,https://ndownloader.figshare.com/files/53993654
9,FILE 10.mat,68.66,https://ndownloader.figshare.com/files/53993660


Total files: 22
Total size (MB): 5282.45


In [ ]:

import os
import requests
from tqdm.auto import tqdm

MAT_FILES = {
    "FILE 1.mat": "https://ndownloader.figshare.com/files/53993651",
    "FILE 2.mat": "https://ndownloader.figshare.com/files/53993645",
    "FILE 3.mat": "https://ndownloader.figshare.com/files/53993648",
    "FILE 4.mat": "https://ndownloader.figshare.com/files/53993657",
    "FILE 5.mat": "https://ndownloader.figshare.com/files/53993666",
    "FILE 6.mat": "https://ndownloader.figshare.com/files/53993642",
    "FILE 7.mat": "https://ndownloader.figshare.com/files/53993663",
    "FILE 8.mat": "https://ndownloader.figshare.com/files/53993669",
    "FILE 9.mat": "https://ndownloader.figshare.com/files/53993654",
    "FILE 10.mat": "https://ndownloader.figshare.com/files/53993660",
    "LABEL DATASET.mat": "https://ndownloader.figshare.com/files/53993639"
}

MAT_DIR = os.path.join(RAW_DIR, "figshare_mat")
os.makedirs(MAT_DIR, exist_ok=True)

def download_file(name, url):
    path = os.path.join(MAT_DIR, name)

    if os.path.exists(path) and os.path.getsize(path) > 0:
        print(f"Already downloaded: {name}")
        return

    temp_path = path + ".part"

    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))

        with open(temp_path, "wb") as f:
            with tqdm(
                total=total if total else None,
                unit="B",
                unit_scale=True,
                desc=name
            ) as pbar:
                for chunk in r.iter_content(chunk_size=1024*1024):
                    if chunk:
                        f.write(chunk)
                        pbar.update(len(chunk))

    os.replace(temp_path, path)

for name, url in MAT_FILES.items():
    download_file(name, url)

print("\nDownload completed!")

for name in MAT_FILES:
    path = os.path.join(MAT_DIR, name)
    print(name, round(os.path.getsize(path)/1024**2, 2), "MB")


FILE 1.mat:   0%|          | 0.00/72.0M [00:00<?, ?B/s]

FILE 2.mat:   0%|          | 0.00/72.0M [00:00<?, ?B/s]

FILE 3.mat:   0%|          | 0.00/72.0M [00:00<?, ?B/s]

FILE 4.mat:   0%|          | 0.00/72.0M [00:00<?, ?B/s]

FILE 5.mat:   0%|          | 0.00/72.0M [00:00<?, ?B/s]

FILE 6.mat:   0%|          | 0.00/72.0M [00:00<?, ?B/s]

... (2 outputs left out)

FILE 9.mat:   0%|          | 0.00/72.0M [00:00<?, ?B/s]

FILE 10.mat:   0%|          | 0.00/72.0M [00:00<?, ?B/s]

LABEL DATASET.mat:   0%|          | 0.00/160k [00:00<?, ?B/s]


Download completed!
FILE 1.mat 68.66 MB
FILE 2.mat 68.66 MB
FILE 3.mat 68.66 MB
FILE 4.mat 68.66 MB
FILE 5.mat 68.66 MB
FILE 6.mat 68.66 MB
FILE 7.mat 68.66 MB
FILE 8.mat 68.66 MB
FILE 9.mat 68.66 MB
FILE 10.mat 68.66 MB
LABEL DATASET.mat 0.15 MB


In [ ]:

import scipy.io
import h5py
import os

def inspect_mat_file(path):
    print("\n" + "="*65)
    print("FILE:", os.path.basename(path))

    try:
        # MATLAB v7.2 and earlier
        variables = scipy.io.whosmat(path)

        for name, shape, dtype in variables:
            print(f"Variable: {name}")
            print(f"Shape: {shape}")
            print(f"Type: {dtype}")

    except (NotImplementedError, ValueError):
        # MATLAB v7.3 uses HDF5
        print("MATLAB v7.3 / HDF5 format detected")

        with h5py.File(path, "r") as f:
            def show_item(name, obj):
                if isinstance(obj, h5py.Dataset):
                    print(
                        f"Variable: {name} | "
                        f"Shape: {obj.shape} | "
                        f"Type: {obj.dtype}"
                    )

            f.visititems(show_item)

for filename in MAT_FILES:
    path = os.path.join(MAT_DIR, filename)
    inspect_mat_file(path)



FILE: FILE 1.mat
Variable: data
Shape: (1000000, 9)
Type: double

FILE: FILE 2.mat
Variable: data
Shape: (1000000, 9)
Type: double

FILE: FILE 3.mat
Variable: data
Shape: (1000000, 9)
Type: double

FILE: FILE 4.mat
Variable: data
Shape: (1000000, 9)
Type: double

FILE: FILE 5.mat
Variable: data
Shape: (1000000, 9)
Type: double

FILE: FILE 6.mat
Variable: data
Shape: (1000000, 9)
Type: double

FILE: FILE 7.mat
Variable: data
Shape: (1000000, 9)
Type: double

FILE: FILE 8.mat
Variable: data
Shape: (1000000, 9)
Type: double

FILE: FILE 9.mat
Variable: data
Shape: (1000000, 9)
Type: double

FILE: FILE 10.mat
Variable: data
Shape: (1000000, 9)
Type: double

FILE: LABEL DATASET.mat
Variable: data
Shape: (1, 1)
Type: struct


In [ ]:

label_path = os.path.join(MAT_DIR, "LABEL DATASET.mat")

print("Inspecting label file:")
inspect_mat_file(label_path)

try:
    labels = scipy.io.loadmat(label_path)

    print("\nLabel variables:")
    for key, value in labels.items():
        if not key.startswith("__"):
            print("\nVariable:", key)
            print("Shape:", getattr(value, "shape", None))
            print("Data type:", getattr(value, "dtype", None))
            print("Sample:", str(value)[:500])

except NotImplementedError:
    print("Label file requires MATLAB v7.3/HDF5 inspection.")


Inspecting label file:

FILE: LABEL DATASET.mat
Variable: data
Shape: (1, 1)
Type: struct

Label variables:

Variable: data
Shape: (1, 1)
Data type: [('label', 'O')]
Sample: [[(array([[ 1,  1,  1, ..., 13, 13, 13]], shape=(1, 19982)),)]]


In [ ]:

import scipy.io
import numpy as np
import pandas as pd
import os

label_path = os.path.join(MAT_DIR, "LABEL DATASET.mat")

mat = scipy.io.loadmat(
    label_path,
    squeeze_me=True,
    struct_as_record=False
)

label_struct = mat["data"]
labels = np.asarray(label_struct.label).flatten()

print("LABEL INVESTIGATION")
print("-" * 45)

print("Total labels:", len(labels))
print("Data type:", labels.dtype)
print("First 30 labels:", labels[:30])

unique_labels, counts = np.unique(
    labels,
    return_counts=True
)

label_summary = pd.DataFrame({
    "Label": unique_labels,
    "Count": counts,
    "Percentage": counts / len(labels) * 100
})

display(label_summary)

print("Number of unique classes:", len(unique_labels))
print("Minimum label:", labels.min())
print("Maximum label:", labels.max())


LABEL INVESTIGATION
---------------------------------------------
Total labels: 19982
Data type: int64
First 30 labels: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]


,Label,Count,Percentage
0,1,1698,8.497648
1,2,99,0.495446
2,3,2197,10.994895
3,4,799,3.998599
4,5,1999,10.004004
5,6,1999,10.004004
6,7,1999,10.004004
7,8,99,0.495446
8,9,2037,10.194175
9,10,1059,5.299770


Number of unique classes: 13
Minimum label: 1
Maximum label: 13


In [ ]:

import scipy.io
import numpy as np
import os

file_path = os.path.join(MAT_DIR, "FILE 1.mat")

mat = scipy.io.loadmat(file_path)
signal = mat["data"]

channel_names = [
    "Acceleration_X",
    "Acceleration_Y",
    "Acceleration_Z",
    "Current_1",
    "Current_2",
    "Current_3",
    "Voltage_1",
    "Voltage_2",
    "Voltage_3"
]

print("Signal shape:", signal.shape)
print("Sampling frequency: 50000 Hz")

signal_summary = pd.DataFrame({
    "Channel": channel_names,
    "Mean": np.mean(signal, axis=0),
    "Std": np.std(signal, axis=0),
    "Min": np.min(signal, axis=0),
    "Max": np.max(signal, axis=0),
    "NaN_Count": np.isnan(signal).sum(axis=0)
})

display(signal_summary)


Signal shape: (1000000, 9)
Sampling frequency: 50000 Hz


,Channel,Mean,Std,Min,Max,NaN_Count
0,Acceleration_X,-0.002526,0.012007,-0.045847,0.039604,0
1,Acceleration_Y,0.005559,0.014466,-0.043187,0.060461,0
2,Acceleration_Z,1.004428,0.011071,0.969181,1.046229,0
3,Current_1,0.013591,0.276676,-1.880093,2.032533,0
4,Current_2,0.010203,0.278279,-1.929533,2.022233,0
5,Current_3,0.015048,0.286613,-2.170553,1.922666,0
6,Voltage_1,9.621985,210.682431,-373.208777,395.430403,0
7,Voltage_2,-0.748956,213.388638,-386.859737,385.807886,0
8,Voltage_3,-10.059125,213.536682,-396.209662,376.122257,0


In [ ]:

import numpy as np
import pandas as pd

# Find consecutive label segments
change_points = np.where(np.diff(labels) != 0)[0] + 1

starts = np.r_[0, change_points]
ends = np.r_[change_points, len(labels)]

runs = pd.DataFrame({
    "Start_Index": starts,
    "End_Index": ends - 1,
    "Label": labels[starts],
    "Window_Count": ends - starts
})

print("Total consecutive label segments:", len(runs))
print("\nFirst 40 label segments:")
display(runs.head(40))

print("\nLast 20 label segments:")
display(runs.tail(20))

# Check expected window counts
window_size = 1000
step_size = 500
samples_per_file = 1_000_000

expected_per_file = (
    (samples_per_file - window_size) // step_size
) + 1

print("\nExpected windows per file:", expected_per_file)
print("Expected total windows:", expected_per_file * 10)
print("Actual labels:", len(labels))
print("Difference:", expected_per_file * 10 - len(labels))

# Inspect labels at hypothetical file boundaries
print("\nLabels around expected recording boundaries:")

for i in range(1, 10):
    boundary = i * expected_per_file
    print(
        f"Boundary {i}:",
        labels[max(0, boundary-3):boundary+3]
    )


Total consecutive label segments: 13

First 40 label segments:


,Start_Index,End_Index,Label,Window_Count
0,0,1697,1,1698
1,1698,1796,2,99
2,1797,3993,3,2197
3,3994,4792,4,799
4,4793,6791,5,1999
5,6792,8790,6,1999
6,8791,10789,7,1999
7,10790,10888,8,99
8,10889,12925,9,2037
9,12926,13984,10,1059



Last 20 label segments:


,Start_Index,End_Index,Label,Window_Count
0,0,1697,1,1698
1,1698,1796,2,99
2,1797,3993,3,2197
3,3994,4792,4,799
4,4793,6791,5,1999
5,6792,8790,6,1999
6,8791,10789,7,1999
7,10790,10888,8,99
8,10889,12925,9,2037
9,12926,13984,10,1059



Expected windows per file: 1999
Expected total windows: 19990
Actual labels: 19982
Difference: 8

Labels around expected recording boundaries:
Boundary 1: [3 3 3 3 3 3]
Boundary 2: [4 4 4 4 4 4]
Boundary 3: [5 5 5 5 5 5]
Boundary 4: [6 6 6 6 6 6]
Boundary 5: [7 7 7 7 7 7]
Boundary 6: [9 9 9 9 9 9]
Boundary 7: [11 11 11 11 11 11]
Boundary 8: [12 12 12 12 12 12]
Boundary 9: [13 13 13 13 13 13]


In [ ]:
import scipy.io
import numpy as np

label_path = os.path.join(MAT_DIR, "LABEL DATASET.mat")

mat = scipy.io.loadmat(
    label_path,
    squeeze_me=True,
    struct_as_record=False
)

label_struct = mat["data"]

print("Fields in label structure:")
print(label_struct._fieldnames)

print("\nField details:")
for field in label_struct._fieldnames:
    value = getattr(label_struct, field)

    print(f"\nField: {field}")
    print("Type:", type(value))

    if isinstance(value, np.ndarray):
        print("Shape:", value.shape)
        print("Dtype:", value.dtype)
        print("First values:", value.flatten()[:10])
    else:
        print("Value:", value)

Fields in label structure:
['label']

Field details:

Field: label
Type: <class 'numpy.ndarray'>
Shape: (19982,)
Dtype: int64
First values: [1 1 1 1 1 1 1 1 1 1]


In [ ]:
import scipy.io as sio
import numpy as np

# Load your MATLAB dataset using the correct path
data = sio.loadmat(label_path, squeeze_me=True, struct_as_record=False)

# Extract labels from the correct structure variable
labels_from_file = data['data'].label.flatten()

# Display unique classes and counts
unique, counts = np.unique(labels_from_file, return_counts=True)

print("Unique labels:", unique)
for label, count in zip(unique, counts):
    print(f"Class {label}: {count} samples")

Unique labels: [ 1  2  3  4  5  6  7  8  9 10 11 12 13]
Class 1: 1698 samples
Class 2: 99 samples
Class 3: 2197 samples
Class 4: 799 samples
Class 5: 1999 samples
Class 6: 1999 samples
Class 7: 1999 samples
Class 8: 99 samples
Class 9: 2037 samples
Class 10: 1059 samples
Class 11: 1999 samples
Class 12: 1999 samples
Class 13: 1999 samples


In [ ]:
for key, value in data.items():
    if not key.startswith('__'):
        print(f"\nField: {key}")
        print("Type:", type(value))

        if isinstance(value, np.ndarray):
            print("Shape:", value.shape)
            print("Dtype:", value.dtype)


Field: data
Type: <class 'scipy.io.matlab._mio5_params.mat_struct'>


In [ ]:
# Inspect fields inside MATLAB structure
mat_data = data['data']

print("Structure fields:")

for field in mat_data._fieldnames:
    value = getattr(mat_data, field)

    print(f"\nField: {field}")
    print("Type:", type(value))

    if isinstance(value, np.ndarray):
        print("Shape:", value.shape)
        print("Dtype:", value.dtype)

Structure fields:

Field: label
Type: <class 'numpy.ndarray'>
Shape: (19982,)
Dtype: int64


In [ ]:
labels = mat_data.label.flatten()

unique, counts = np.unique(labels, return_counts=True)

print("Unique labels:", unique)
print("Total samples:", len(labels))

Unique labels: [ 1  2  3  4  5  6  7  8  9 10 11 12 13]
Total samples: 19982


In [ ]:
print("MATLAB structure:")
print(mat_data)

print("\nAll structure fields:")
print(mat_data._fieldnames)

print("\nMAT-file top-level variables:")
for key in data.keys():
    print(key)

MATLAB structure:

All structure fields:
['label']

MAT-file top-level variables:
__header__
__version__
__globals__
data


In [ ]:
import os
import numpy as np
import scipy.io as sio
from tqdm.auto import tqdm

# Configure window parameters
window_size = 1000
step_size = 500
num_files = 10
expected_windows_per_file = 1999

all_windows = []
all_labels = []

print("Processing files and extracting windows...")
for i in range(1, num_files + 1):
    file_name = f"FILE {i}.mat"
    file_path = os.path.join(MAT_DIR, file_name)
    print(f"Loading {file_name}...")

    mat = sio.loadmat(file_path)
    signal_data = mat['data'] # Shape: (1000000, 9)

    # Corresponding index range in the labels array
    start_label_idx = (i - 1) * expected_windows_per_file
    end_label_idx = i * expected_windows_per_file
    file_labels = labels[start_label_idx:end_label_idx]

    # Extract sliding windows
    num_actual_windows = min(expected_windows_per_file, len(file_labels))
    for w_idx in range(num_actual_windows):
        start_sample = w_idx * step_size
        end_sample = start_sample + window_size

        window = signal_data[start_sample:end_sample, :]
        all_windows.append(window)
        all_labels.append(file_labels[w_idx])

X = np.array(all_windows, dtype=np.float32)
y = np.array(all_labels, dtype=np.int64)

print(f"\nExtraction complete!")
print("Feature tensor shape (X):", X.shape)
print("Label array shape (y):", y.shape)

# Save dataset to processed folder
np.save(os.path.join(PROCESSED_DIR, "X_windows.npy"), X)
np.save(os.path.join(PROCESSED_DIR, "y_labels.npy"), y)
print("Dataset successfully saved to:", PROCESSED_DIR)

Processing files and extracting windows...
Loading FILE 1.mat...
Loading FILE 2.mat...
Loading FILE 3.mat...
Loading FILE 4.mat...
Loading FILE 5.mat...
Loading FILE 6.mat...
Loading FILE 7.mat...
Loading FILE 8.mat...
Loading FILE 9.mat...
Loading FILE 10.mat...

Extraction complete!
Feature tensor shape (X): (19982, 1000, 9)
Label array shape (y): (19982,)
Dataset successfully saved to: /content/drive/MyDrive/Motor_Fault_Analysis/datasets/processed


In [ ]:
import os
import numpy as np

# Verify saved dataset
x_path = os.path.join(PROCESSED_DIR, "X_windows.npy")
y_path = os.path.join(PROCESSED_DIR, "y_labels.npy")

if os.path.exists(x_path) and os.path.exists(y_path):
    X_loaded = np.load(x_path, mmap_mode='r')
    y_loaded = np.load(y_path)

    print("--- Verification Successful ---")
    print("Loaded X shape:", X_loaded.shape, "| dtype:", X_loaded.dtype)
    print("Loaded y shape:", y_loaded.shape, "| dtype:", y_loaded.dtype)
    print("Unique labels in y:", np.unique(y_loaded))
else:
    print("Processed files not found. Please re-run the window extraction cell.")

--- Verification Successful ---
Loaded X shape: (19982, 1000, 9) | dtype: float32
Loaded y shape: (19982,) | dtype: int64
Unique labels in y: [ 1  2  3  4  5  6  7  8  9 10 11 12 13]


In [ ]:
import os

for root, dirs, files_list in os.walk('/content'):
    for file in files_list:
        print(os.path.join(root, file))

/content/.config/gce
/content/.config/.last_update_check.json
/content/.config/.last_opt_in_prompt.yaml
/content/.config/active_config
/content/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
/content/.config/config_sentinel
/content/.config/.last_survey_prompt.yaml
/content/.config/default_configs.db
/content/.config/logs/2026.10.21/13.28.05.116687.log
/content/.config/logs/2026.10.21/13.27.57.480371.log
/content/.config/logs/2026.10.21/13.28.04.564766.log
/content/.config/logs/2026.10.21/13.27.58.650478.log
/content/.config/logs/2026.10.21/13.27.51.774504.log
/content/.config/logs/2026.10.21/13.27.28.750982.log
/content/.config/configurations/config_default
/content/drive/MyDrive/17411067372158621111981510807561.jpg
/content/drive/MyDrive/IMG-20240508-WA0039.jpg
/content/drive/MyDrive/Hustle.x T & C.zip
/content/drive/MyDrive/IMG_20240630_232802.JPG
/content/drive/MyDrive/WhatsApp Image 2024-07-24 at 10.50.10 PM.jpeg
/content/drive/MyDrive/IMG-20240827-WA0040.j

In [ ]:
import numpy as np
import os

base_path = "/content/drive/MyDrive/Motor_Fault_Analysis/datasets/processed"

X = np.load(os.path.join(base_path, "X_windows.npy"))
y = np.load(os.path.join(base_path, "y_labels.npy"))

print("X shape:", X.shape)
print("X dtype:", X.dtype)

print("\ny shape:", y.shape)
print("y dtype:", y.dtype)

print("\nUnique labels:", np.unique(y))
print("\nFirst 5 samples:")
print(X[:5])

X shape: (19982, 1000, 9)
X dtype: float32

y shape: (19982,)
y dtype: int64

Unique labels: [ 1  2  3  4  5  6  7  8  9 10 11 12 13]

First 5 samples:
[[[ 4.8244423e-03  5.1823873e-03  1.0091555e+00 ...  9.6154060e+00
   -5.2592546e-01 -9.9979248e+00]
  [ 4.8244423e-03  5.1823873e-03  1.0091555e+00 ...  9.6154060e+00
   -5.2592546e-01 -9.9979248e+00]
  [-5.8519243e-05  5.1823873e-03  1.0061038e+00 ...  9.4933319e+00
   -4.3436995e-01 -1.0425184e+01]
  ...
  [ 4.8244423e-03  6.9098501e-03  1.0061038e+00 ...  9.6459246e+00
   -7.3955506e-01 -1.0364147e+01]
  [ 4.8244423e-03  6.9098501e-03  1.0061038e+00 ...  9.6459246e+00
   -7.3955506e-01 -1.0364147e+01]
  [ 4.8244423e-03  6.9098501e-03  1.0061038e+00 ...  9.6459246e+00
   -7.3955506e-01 -1.0364147e+01]]

 [[-3.7207403e-03 -5.1823873e-03  1.0000000e+00 ...  9.7374802e+00
   -5.5644399e-01 -9.9063692e+00]
  [-3.7207403e-03 -5.1823873e-03  1.0000000e+00 ...  9.7374802e+00
   -5.5644399e-01 -9.9063692e+00]
  [-4.9414807e-03 -1.7274625e-03

In [ ]:
print("Number of input samples:", len(X))
print("Number of labels:", len(y))

if len(X) == len(y):
    print("SUCCESS: X and y are aligned!")
else:
    print("WARNING: Number of samples and labels do not match!")

Number of input samples: 19982
Number of labels: 19982
SUCCESS: X and y are aligned!


In [ ]:
print("X shape:", X.shape)
print("X dtype:", X.dtype)

print("y shape:", y.shape)
print("Unique classes:", np.unique(y))

X shape: (19982, 1000, 9)
X dtype: float32
y shape: (19982,)
Unique classes: [ 1  2  3  4  5  6  7  8  9 10 11 12 13]


In [ ]:
print("\nFirst sample:")
print(X[0])

print("\nMinimum value:", np.min(X))
print("Maximum value:", np.max(X))
print("Contains NaN:", np.isnan(X).any())
print("Contains Inf:", np.isinf(X).any())


First sample:
[[ 4.8244423e-03  5.1823873e-03  1.0091555e+00 ...  9.6154060e+00
  -5.2592546e-01 -9.9979248e+00]
 [ 4.8244423e-03  5.1823873e-03  1.0091555e+00 ...  9.6154060e+00
  -5.2592546e-01 -9.9979248e+00]
 [-5.8519243e-05  5.1823873e-03  1.0061038e+00 ...  9.4933319e+00
  -4.3436995e-01 -1.0425184e+01]
 ...
 [ 4.8244423e-03  6.9098501e-03  1.0061038e+00 ...  9.6459246e+00
  -7.3955506e-01 -1.0364147e+01]
 [ 4.8244423e-03  6.9098501e-03  1.0061038e+00 ...  9.6459246e+00
  -7.3955506e-01 -1.0364147e+01]
 [ 4.8244423e-03  6.9098501e-03  1.0061038e+00 ...  9.6459246e+00
  -7.3955506e-01 -1.0364147e+01]]

Minimum value: -625.9112
Maximum value: 515.7862
Contains NaN: False
Contains Inf: False


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
from scipy.signal import periodogram

sampling_rate = 50000  # Verify against dataset documentation

def extract_features(window, fs=50000):
    features = []

    for ch in range(window.shape[1]):
        signal = window[:, ch]

        # Time-domain features
        mean = np.mean(signal)
        std = np.std(signal)
        rms = np.sqrt(np.mean(signal ** 2))
        minimum = np.min(signal)
        maximum = np.max(signal)
        peak_to_peak = maximum - minimum
        variance = np.var(signal)
        skewness = skew(signal)
        kurt = kurtosis(signal)
        crest_factor = np.max(np.abs(signal)) / (rms + 1e-10)

        # Frequency-domain features
        frequencies, power = periodogram(signal, fs=fs)

        dominant_frequency = frequencies[np.argmax(power[1:]) + 1]
        spectral_energy = np.sum(power)

        features.extend([
            mean, std, rms, minimum, maximum,
            peak_to_peak, variance, skewness,
            kurt, crest_factor,
            dominant_frequency, spectral_energy
        ])

    return features


# Extract features from all signal windows
X_features = np.array([
    extract_features(window, sampling_rate)
    for window in X
], dtype=np.float32)

# Replace any undefined numerical values
X_features = np.nan_to_num(X_features)

print("Original shape:", X.shape)
print("Extracted features shape:", X_features.shape)
print("Labels shape:", y.shape)

Original shape: (19982, 1000, 9)
Extracted features shape: (19982, 108)
Labels shape: (19982,)


In [ ]:
processed_path = "/content/drive/MyDrive/Motor_Fault_Analysis/datasets/processed"

np.save(processed_path + "/X_features.npy", X_features)

df = pd.DataFrame(X_features)
df["label"] = y

df.to_csv(
    processed_path + "/motor_fault_features.csv",
    index=False
)

print("Feature dataset saved successfully!")

Feature dataset saved successfully!


In [ ]:
import pandas as pd
import numpy as np

csv_path = "/content/drive/MyDrive/Motor_Fault_Analysis/datasets/processed/motor_fault_features.csv"

df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

print("\nClass distribution:")
print(df["label"].value_counts().sort_index())

Dataset shape: (19982, 109)

First 5 rows:


,0,1,2,3,4,5,6,7,8,9,...,99,100,101,102,103,104,105,106,107,label
0,-0.000864,0.003222,0.003335,-0.007383,0.006045,0.013428,0.000010,0.147287,-0.749935,2.213480,...,-10.486221,-9.570665,0.915556,0.022724,0.199177,0.138114,1.036007,2250.0,0.000454,1
1,-0.000795,0.003289,0.003383,-0.007383,0.006045,0.013428,0.000011,0.273483,-0.843233,2.182204,...,-10.577776,-9.601184,0.976592,0.026907,0.061597,0.311748,1.045435,2250.0,0.000538,1
2,-0.000872,0.003204,0.003321,-0.006162,0.006045,0.012207,0.000010,0.230800,-0.900587,1.855640,...,-10.577776,-9.601184,0.976592,0.026109,0.046892,0.428419,1.046212,2250.0,0.000522,1
3,-0.000944,0.003129,0.003268,-0.006162,0.006045,0.012207,0.000010,0.142208,-0.904739,1.885606,...,-10.516740,-9.662221,0.854519,0.021947,-0.127646,-0.279057,1.039641,2250.0,0.000439,1
4,-0.000919,0.003166,0.003297,-0.007383,0.006045,0.013428,0.000010,0.184805,-0.783113,2.239226,...,-10.516740,-9.631702,0.885037,0.022722,-0.093350,-0.136303,1.039665,2250.0,0.000454,1



Missing values: 0
Duplicate rows: 0

Class distribution:
label
1     1698
2       99
3     2197
4      799
5     1999
6     1999
7     1999
8       99
9     2037
10    1059
11    1999
12    1999
13    1999
Name: count, dtype: int64


In [ ]:
import pandas as pd

df = pd.read_csv(
    "/content/drive/MyDrive/Motor_Fault_Analysis/datasets/processed/motor_fault_features.csv"
)

X = df.drop(columns=["label"])
y = df["label"]

print(X.shape)
print(y.shape)

(19982, 108)
(19982,)
